# 00 - Data Pull

Ingestion for all five leagues. Two cadences:

| Cadence | When | Steps |
|---|---|---|
| `mode="current"` | before each prediction run | current-season results, match stats, fixtures, rest-day update |
| `mode="all"` | first build / start of season | everything, including the full history |

The notebook owns no ingestion logic -- it calls `fpp.ingest.refresh`, the same
functions the Run notebook uses. Nothing here is duplicated anywhere else.

In [1]:
# The package is installed editable (`pip install -e .`), so this works from any
# working directory -- no `os.getcwd()` gymnastics.
import fpp
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("fpp", fpp.__version__, "| targets:", fpp.TARGETS)

fpp 0.1.0 | targets: ('goals', 'shots', 'sot', 'corners')


## 1. Configuration

`DATE_FROM` / `DATE_TO` are `dd-mm-yyyy` and bound the *fixture* pull only. Leave
them as `None` to skip fixtures and refresh results alone.

`CURRENT_SEASON` / `PREVIOUS_SEASON` are the only two things to change at a season
rollover. Everything else derives from them -- the history's upper bound, the ESPN
season label, the held-out test boundary and the first CV validation season. They
are validated on the way in, so a mismatched pair fails here rather than surfacing
later as a quietly wrong split.

In [2]:
MODE            = "all"         # "current" (day to day) | "all" (start of season)
DATE_FROM       = None          # e.g. "20-05-2026"
DATE_TO         = None          # e.g. "24-05-2026"
FORCE           = False         # re-resolve ESPN team ids from scratch

PREVIOUS_SEASON = "2025/2026"   # last completed season -- upper bound of the history
CURRENT_SEASON  = "2026/2027"   # this season

# Second-tier seasons to cache ESPN stats for. Derived from the rollover above
# rather than listed, so next season extends it by itself and nothing here needs
# editing. Seven covers `PROMOTED_LOOKBACK` (5) plus the promotion season being
# measured, which is what `fpp.promoted` reads.
LOWER_SEASONS   = fpp.config.historic_seasons()[-7:]
LOWER_BACKFILL  = True         # False = report the cost only; True = actually fetch

fpp.config.set_seasons(current=CURRENT_SEASON, previous=PREVIOUS_SEASON)
fpp.paths.ensure_dirs()

print("Inputs :", fpp.paths.INPUTS)
print("Cache  :", fpp.paths.CACHE_ROOT, "(outside iCloud, by design)")
print()
print("current  :", fpp.config.current_season(), f"({fpp.config.current_season_label()})")
print("history  :", fpp.config.historic_seasons()[0], "..", fpp.config.historic_seasons()[-1],
      f"({len(fpp.config.historic_seasons())} seasons)")
print("test     :", fpp.config.test_season(), "(held out, scored once)")
print("first val:", fpp.config.first_val_season())

Inputs : /Users/patrickknott/Desktop/Football_Prediction_Project V2/Inputs
Cache  : /Users/patrickknott/.cache/football_prediction (outside iCloud, by design)

current  : 2026/2027 (2026-27)
history  : 2014/2015 .. 2025/2026 (12 seasons)
test     : 2025/2026 (held out, scored once)
first val: 2020/2021


## 2. Run the pull

Every step is independently guarded -- one failing source does not abort the rest,
and the report at the end says exactly what ran.

In [3]:
report = fpp.ingest.refresh(MODE, date_from=DATE_FROM, date_to=DATE_TO, force=FORCE)
print()
print(report)

[understat_history]
  Prem...


[08/25/26 00:00:11] INFO     No custom team name replacements found. You can configure these in       _config.py:91
                             /Users/patrickknott/soccerdata/config/teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:189
                             /Users/patrickknott/soccerdata/config/league_dict.json.                               

  Liga...
  Bund...
  Serie...
  Ligue...
[understat_current]
  Bund 2026/2027: cached file unusable -- refetching


[08/25/26 00:00:12] INFO     Saving cached data to                                                   _common.py:250
                             /Users/patrickknott/Desktop/Football_Prediction_Project                               
                             V2/Inputs/soccerdata_cache                                                            

...

[08/25/26 00:00:47] INFO     Saving cached data to                                                   _common.py:250
                             /Users/patrickknott/Desktop/Football_Prediction_Project                               
                             V2/Inputs/soccerdata_cache                                                            

                    INFO     Saving cached data to                                                   _common.py:250
                             /Users/patrickknott/Desktop/Football_Prediction_Project                               
                             V2/Inputs/soccerdata_cache                                                            

  Bund 2026/2027: cached file unusable -- refetching


[08/25/26 00:00:48] INFO     Saving cached data to                                                   _common.py:250
                             /Users/patrickknott/Desktop/Football_Prediction_Project                               
                             V2/Inputs/soccerdata_cache                                                            

  Bund 2026/2027: no usable rows (the season is listed but returned no matches -- none played yet?) -- not cached, will refetch next run
  Bund 2026/2027: no usable rows -- D1 Current Season.csv kept existing file
  dropped 3 unplayed/incomplete rows
  ESPN: fetching 8 missing date file(s) across 4 league-season(s)
  Liga 2627: 2 dates missing, 2 new files
  Ligue 2627: 2 dates missing, 2 new files
  Prem 2627: 2 dates missing, 2 new files
  Serie 2627: 2 dates missing, 2 new files
  Dropped 1,444 event(s) with all-zero stats (ESPN empty-block sentinel, not a 0-0 result)
  Dropped 1,050 event(s) with every shot on target and no corners (malformed ESPN stat block, not a played match)
  Parsed 30,679 events with full shots/SOT/corners from the cache
  91 second-tier-only club(s) unmapped -- expected, Understat does not cover those divisions; their matches are still kept for the mapped opponent
  Saved 28,951 rows -> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Inputs/Stats/

## 3. Second-tier scoreboards (promoted-club history)

`fpp.promoted` scales a promoted club's cold-start seed by what it actually did in
the division below, which needs ESPN stats for the five second tiers. Understat
does not cover them, so unlike everything above there is no fixture list to drive
the pull from -- `missing_dates` reads the Understat match table and finds nothing
there. This enumerates the season window instead and lets the cache do the rest.

**Run it once and it stays cheap.** Existing date files are never refetched, so a
cold cache is ~10,650 requests (about 90 minutes) and every run after that only
picks up what is new -- one extra season at a rollover is roughly 1,500 files, and
a mid-season re-run is a few dozen. Interrupting is free; it resumes.

Set `LOWER_BACKFILL = True` above to fetch. Left `False` it only reports the cost.

Two things worth knowing about the data:

* **ESPN's Championship stats fail in 2024/25** -- sampled at 1 real match in 49,
  the rest carrying the all-zero placeholder. `_is_sentinel` rejects those at parse
  time, so that season contributes nothing to the promoted benchmark rather than
  poisoning it with zeros. Neighbouring seasons are complete, and 2025/26 sampled
  100% real across all five tiers.
* Clubs that are never promoted go unmapped and are dropped. That costs nothing
  here: only promoted clubs and the promoted cohort they are measured against are
  ever read, and all 104 in the data are already in the ESPN name map.

In [4]:
n = fpp.ingest.scoreboard.backfill_lower_leagues(
    LOWER_SEASONS, dry_run=not LOWER_BACKFILL)

print()
print("seasons  :", LOWER_SEASONS[0], "..", LOWER_SEASONS[-1])
print("outstanding date files:" if not LOWER_BACKFILL else "files fetched:", f"{n:,}")
if not LOWER_BACKFILL and n:
    print("\nset LOWER_BACKFILL = True above to fetch them")


seasons  : 2019/2020 .. 2025/2026
files fetched: 0


## 4. Shots, shots on target and corners

These come from ESPN **scoreboard** JSON that `soccerdata` already caches locally,
so the historical pull is a re-parse with zero HTTP calls -- not the per-fixture
boxscore scrape originally envisaged.

`soccerdata` does not expose these fields itself: `read_schedule` parses fixture
metadata only, and `read_matchsheet` uses a different endpoint that would need one
request per match.

In [5]:
stats = fpp.ingest.scoreboard.build_match_stats(save=True)
print(f"{len(stats):,} fixtures with shots / SOT / corners")
print(f"date range: {stats['date'].min().date()} -> {stats['date'].max().date()}")
stats.groupby("league_key").size().to_frame("fixtures")

  Dropped 1,444 event(s) with all-zero stats (ESPN empty-block sentinel, not a 0-0 result)
  Dropped 1,050 event(s) with every shot on target and no corners (malformed ESPN stat block, not a played match)
  Parsed 30,679 events with full shots/SOT/corners from the cache
  91 second-tier-only club(s) unmapped -- expected, Understat does not cover those divisions; their matches are still kept for the mapped opponent
  Saved 28,951 rows -> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Inputs/Stats/espn_match_stats.csv
28,951 fixtures with shots / SOT / corners
date range: 2013-08-24 -> 2026-08-17


,fixtures
league_key,
Bund,3622
Bund2,1598
Liga,4513
Liga2,1787
Ligue,4186
Ligue2,516
Prem,4042
Prem2,2685
Serie,4838


## 5. Verification

The pull above already reconciled: for every `(league, season)` that should exist it
checked whether real, complete data is there and fetched whatever wasn't. This cell
confirms the result rather than asking you to spot a problem.

Two things are checked separately, because they fail separately:

- **Understat** — a season is complete at `teams × (teams − 1)` matches, derived from
  the teams present rather than configured. A shortfall here means missing fixtures.
- **ESPN** — coverage of those matches by the scoreboard stats.

`audit()` fetches nothing, so it is safe to re-run at any point. Anything listed as
short after a reconcile is a genuine gap in the source, not something to re-run.

In [6]:
state = fpp.reconcile.audit()

print(f"{len(state)} league-season(s) expected "
      f"({len(fpp.config.historic_seasons())} historic + current, x{len(fpp.LEAGUES)} leagues)")
print(f"Understat matches : {state['matches'].sum():,} / {state['expected'].sum():,} expected")
print(f"ESPN coverage     : {state['with_stats'].sum():,} / {state['matches'].sum():,} "
      f"({100 * state['with_stats'].sum() / max(1, state['matches'].sum()):.2f}%)")

short = state[(state["shortfall"] > 0) | (state["pct"] < fpp.reconcile.MIN_COVERAGE)]
if len(short):
    print(f"\n{len(short)} league-season(s) still short:")
    display(short[["league_key", "season", "matches", "expected", "shortfall",
                   "pct", "missing_dates", "action"]])
else:
    print("\nevery league-season complete on Understat and at "
          f"{fpp.reconcile.MIN_COVERAGE:.0f}%+ ESPN coverage")

  dropped 3 unplayed/incomplete rows
  ESPN stats joined to 20,807/21,634 matches (96.2%)
65 league-season(s) expected (12 historic + current, x5 leagues)
Understat matches : 21,634 / 23,136 expected
ESPN coverage     : 18,802 / 21,634 (86.91%)

18 league-season(s) still short:


,league_key,season,matches,expected,shortfall,pct,missing_dates,action
2,Prem,2016/2017,380,380,0,4.7,0,ok
7,Prem,2021/2022,380,380,0,92.9,0,ok
8,Prem,2022/2023,380,380,0,78.2,0,ok
11,Prem,2025/2026,380,380,0,0.0,0,ok
12,Prem,2026/2027,10,380,370,0.0,0,short
21,Liga,2022/2023,380,380,0,93.9,0,ok
24,Liga,2025/2026,380,380,0,0.0,0,ok
25,Liga,2026/2027,16,380,364,0.0,0,short
33,Bund,2021/2022,306,306,0,89.2,0,ok
37,Bund,2025/2026,306,306,0,0.0,0,ok
